In [5]:
import io
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import ipywidgets as widgets

from PIL import Image
from IPython.display import display
from tensorflow.keras.models import load_model
from google.colab import drive

# Incluir el drive en mi path
drive.mount('/content/drive')

# Cargar los modelos
model_original = load_model(
    '/content/drive/MyDrive/8vo/IA/SaveModels/road_signs_model.h5'
)

model_paper = load_model(
    '/content/drive/MyDrive/8vo/IA/SaveModels/road_signs_model_paper.h5'
)

model_hybrid = load_model(
    '/content/drive/MyDrive/8vo/IA/SaveModels/road_signs_model_hybrid.h5'
)


# Clases
classes = [
    'Keep Left',
    'Keep Right',
    'No Entry',
    'Pedestrian Crossing',
    'Stop Sign',
    'Turn Left',
    'Turn Right'
]

# Crear interfaz
def crear_interfaz(modelo, titulo):

    uploader = widgets.FileUpload(
        accept='.png,.jpg,.jpeg',
        multiple=False,
        description='Subir imagen',
        button_style='info'
    )

    output_area = widgets.Output()

    def al_subir_imagen(change):

        with output_area:

            output_area.clear_output()

            if not uploader.value:
                return

            uploaded_filename = list(
                uploader.value.keys()
            )[0]

            content = uploader.value[
                uploaded_filename
            ]['content']

            img = Image.open(
                io.BytesIO(content)
            ).convert('RGB')

            # Preprocesamiento
            img_resized = img.resize((224, 224))

            x = np.array(img_resized) / 255.0
            x = np.expand_dims(x, axis=0)

            # Predicción
            predictions = modelo.predict(
                x,
                verbose=0
            )[0]

            idx_ganador = np.argmax(
                predictions
            )

            pred_label = classes[
                idx_ganador
            ]

            porcentaje_max = (
                predictions[idx_ganador] * 100
            )

            # Visualización
            fig, axes = plt.subplots(
                1,
                2,
                figsize=(13, 5)
            )

            # Imagen
            axes[0].imshow(img)
            axes[0].axis('off')
            axes[0].set_title(
                f'{titulo}\n{uploaded_filename}'
            )

            # Barras
            porcentajes = predictions * 100

            colores = [
                '#2ecc71'
                if i == idx_ganador
                else '#3498db'
                for i in range(len(classes))
            ]

            barras = axes[1].barh(
                classes,
                porcentajes,
                color=colores
            )

            axes[1].invert_yaxis()

            axes[1].set_xlim(0, 110)

            axes[1].set_title(
                f'{pred_label} ({porcentaje_max:.2f}%)'
            )

            axes[1].set_xlabel(
                'Confianza (%)'
            )

            for barra in barras:

                ancho = barra.get_width()

                axes[1].text(
                    ancho + 1,
                    barra.get_y()
                    + barra.get_height()/2,
                    f'{ancho:.1f}%',
                    va='center'
                )

            plt.tight_layout()
            plt.show()

    uploader.observe(
        al_subir_imagen,
        names='value'
    )

    return widgets.VBox([
        widgets.HTML(
            f'<h2>{titulo}</h2>'
        ),
        uploader,
        output_area
    ])

# Interfaces de cada modelo
tab1 = crear_interfaz(
    model_original,
    "Modelo Original"
)

tab2 = crear_interfaz(
    model_paper,
    "Modelo Paper"
)

tab3 = crear_interfaz(
    model_hybrid,
    "Modelo Hybrid"
)

# Tabs
tabs = widgets.Tab(
    children=[
        tab1,
        tab2,
        tab3
    ]
)

tabs.set_title(0, 'Mi CNN')
tabs.set_title(1, 'CNN Paper')
tabs.set_title(2, 'Híbrido')

display(tabs)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
